<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Breast Cancer Challenge — Optional
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## How to use this notebook

**Optional, about 45 minutes, nothing to submit.** Session 7's methods on a small open dataset: build the best classifier you can for whether a breast mass is malignant, then see what clustering and PCA find in the same features without being told the answer.

The tasks are the ones you know: **» Investigate** a working cell, **» Modify** one thing, and **» Make** your own. **What you should find** blocks fold open: try first.

* The data are the [Wisconsin Diagnostic Breast Cancer](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) dataset, which comes with scikit-learn, under a [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) licence, so you may plot them and share your outputs. MIMIC rules still apply to every other notebook.
* Each row is one breast mass. The 30 features describe its cell nuclei, measured from a digitised image of a fine-needle aspirate: the mean, the standard error and the worst value of ten measurements, such as radius and texture.
* As in the session, every choice is made by cross-validation on the training rows, and the test rows are used once, at the end.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, train_test_split

# scikit-learn steps return tables with their column names, not bare arrays
sklearn.set_config(transform_output='pandas')

data = load_breast_cancer(as_frame=True)
X = data.data

# scikit-learn codes benign as 1: turn it round, so 1 is malignant
malignant = 1 - data.target

X_train, X_test, y_train, y_test = train_test_split(X, malignant, test_size=0.2, stratify=malignant, random_state=0)

# One row per patient, so ordinary folds are enough; stratified keeps the share malignant the same in each
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

print(X.shape, ' malignant:', round(malignant.mean(), 3))
print(len(X_train), 'training rows,', len(X_test), 'test rows')

## 1. A first model
**» Investigate** — Logistic regression on two features, cross-validated on the training rows. **Before you go further: which measure should we judge the models by, accuracy or AUC?** About 37% of the masses are malignant. What would a model that calls every mass benign score on each?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

columns = ['mean radius', 'mean texture']

pipeline = make_pipeline(StandardScaler(), LogisticRegression())
scores = cross_validate(pipeline, X_train[columns], y_train, cv=folds, scoring=['accuracy', 'roc_auc'])
print('Accuracy', round(scores['test_accuracy'].mean(), 3), ' AUC', round(scores['test_roc_auc'].mean(), 3))

<details><summary><b>What you should find</b></summary>

Accuracy **0.873**, AUC **0.943**. Calling every mass benign would score an accuracy of 0.63 and an AUC of 0.5, so accuracy flatters a useless model less here than in MIMIC, where 89% survive; both are worth reporting. Note too that a missed cancer costs far more than a false alarm: accuracy counts them the same.

</details>

## 2. The challenge
**» Make** — **Build the best classifier you can.** 98% cross-validated accuracy is possible. Start from the cell above and change one thing at a time: more features, then other models from the session (`RandomForestClassifier`, `HistGradientBoostingClassifier`), then tune the best with `GridSearchCV` and `cv=folds`. When you have chosen, test it **once** on `X_test`. In a comment: your model, its cross-validated and test accuracy, and why you chose it.

<details><summary><b>Hint</b></summary>

`X_train` on its own has all 30 features. Session 7's Part 2 has the search pattern; here there are no `groups`, because each patient has one row. For logistic regression, try `C` from 0.01 to 100.

</details>

In [ ]:
# Your model, its cross-validated and test accuracy, and why you chose it:

# Your code here

<details><summary><b>What you should find</b></summary>

| Model, all 30 features | CV accuracy | CV AUC |
| :-- | --: | --: |
| Logistic regression, scaled | **0.982** | 0.992 |
| Logistic regression, not scaled | 0.947 | 0.989 |
| Random forest, 300 trees | 0.956 | 0.993 |
| Gradient boosting | 0.960 | 0.991 |

**Scaled logistic regression wins.** Tuning `C` from 0.01 to 100 chooses the default, 1 (0.982); tested once, accuracy **0.974** and AUC 0.993. The forest and boosting have no advantage here: with 455 training rows and features that separate the classes almost along a straight line, there are no bends for trees to find, and the simplest model is also the best. Ensembles win on some tables, not all: compare before you choose. Resampling is not needed either: 37% malignant is not rare.

</details>

## 3. Clusters without the answer
**» Modify** — k-means on all 30 features, scaled, with no diagnosis. The table counts each cluster's masses by their real diagnosis. **Change `n_clusters` to 3.** In a comment: do the clusters match the diagnoses? What might the third cluster be?

In [ ]:
# Do the clusters match the diagnoses, and what might the third cluster be:
from sklearn.cluster import KMeans

scaler = StandardScaler()
scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2, random_state=0)
cluster = kmeans.fit_predict(scaled)

diagnosis = malignant.map({1: 'malignant', 0: 'benign'})
pd.crosstab(cluster, diagnosis)

<details><summary><b>What you should find</b></summary>

With two clusters, one holds 344 benign and 37 malignant masses, the other 175 malignant and 13 benign: k-means, never shown a diagnosis, separates them with 91% agreement. With three, one cluster is **entirely malignant** (118), one is mostly benign (321 with 33 malignant), and a third is **mixed** (36 benign, 61 malignant). The mixed cluster holds the masses whose nuclei look neither clearly benign nor clearly malignant: the hard cases, which a clinician might want to investigate, and where any classifier makes most of its mistakes. Without scaling, the clusters follow the area columns, which run into the thousands, and the agreement is worse.

</details>

## 4. How many features do you need?
**» Make** — **Plot cross-validated accuracy against the number of principal components.** Put `PCA(n_components=n)` between the scaler and logistic regression in a pipeline, and loop over `n` from 1 to 30. In a comment: how few components keep almost all the accuracy?

<details><summary><b>Hint</b></summary>

`make_pipeline(StandardScaler(), PCA(n_components=n), LogisticRegression(max_iter=5000))`. Collect the mean accuracy for each `n` in a list, then `plt.plot`. Fitting `PCA()` to `scaled` from section 3, then `pca.explained_variance_ratio_.cumsum()`, shows how much of the variation each number of components keeps.

</details>

In [ ]:
# How few components keep almost all the accuracy:

# Your code here

<details><summary><b>What you should find</b></summary>

| Components | CV accuracy |
| --: | --: |
| 1 | 0.921 |
| 2 | 0.954 |
| 5 | 0.969 |
| 10 | 0.980 |
| 30 | 0.982 |

One component, a single weighted sum of all 30 features, already gives 92%; ten keep almost all of the accuracy, and hold 95% of the variation. The 30 features overlap heavily (radius, perimeter and area measure the same thing three ways), so most of them add little. PCA is fitted inside each fold here, in the pipeline, as any step that learns from rows must be.

</details>